# 构建三个不同风格空间的 PCFG 模型

## 加载风格空间（训练集）

In [1]:
from pathlib import Path
from typing import List
import json

MUICE_PATH = Path("../Dataset/Muice/train.jsonl")
HARUHI_PATH = Path(r"..\Dataset\Haruhi\Haruhi_clean.jsonl")
PSYDC_PATH1 = Path(r"..\Dataset\PsyDTCorpus\PsyDTCorpus_train_mulit_turn_packing.json")
PSYDC_PATH2 = Path(r"..\Dataset\PsyDTCorpus\PsyDTCorpus_test_single_turn_split.json")

# Check if Dataset exist
assert MUICE_PATH.is_file(), "请确保 Muice Dataset 的路径正确！"
assert HARUHI_PATH.is_file(), "请确保 Haruhi Dataset 的路径正确！"
assert PSYDC_PATH1.is_file(), "请确保 PsyDTCorpus Dataset 的路径正确！"

# Load Muice Dataset
dataset_lines = MUICE_PATH.read_text(encoding="utf-8").splitlines()

muice_responses: List[str] = []

for line in dataset_lines:
    item = json.loads(line)
    muice_responses.append(item["Response"])

# 取 80% 作为训练集
muice_responses = muice_responses[:int(len(muice_responses)*0.8)]

# Load Haruhi Dataset
dataset_lines = HARUHI_PATH.read_text(encoding="utf-8").splitlines()

haruhi_responses: List[str] = []

for line in dataset_lines:
    item = json.loads(line)

    # 只提取单一角色的训练集
    if item["agent_role"] == "神里绫华":
        haruhi_responses.append(item["agent_response"])

# 取 70 %作为训练集
haruhi_responses = haruhi_responses[:int(len(haruhi_responses)*0.7)]


# Load PsyDTCorpus Dataset
psydc_part1 = json.loads(PSYDC_PATH1.read_text(encoding="utf-8"))
psydc_part2 = json.loads(PSYDC_PATH2.read_text(encoding="utf-8"))
psydc: list[dict] = psydc_part1 + psydc_part2
psydc_responses: List[str] = []

for item in psydc:
    messages: list[dict[str, str]] = item["messages"]
    for index in range(2, len(messages), 2):
        message = messages[index]
        assert message["role"] == "assistant", message
        psydc_responses.append(message["content"])

# 取 70 %作为训练集
psydc_responses = psydc_responses[:int(len(psydc_responses)*0.7)]

# 输出所有训练集的长度

print(f"Muice Responses: {len(muice_responses)}")
print(f"Haruhi Responses: {len(haruhi_responses)}")
print(f"PsyDTCorpus Responses: {len(psydc_responses)}")

Muice Responses: 2721
Haruhi Responses: 991
PsyDTCorpus Responses: 89652


## 调用 HanLP 进行成分语法分析

In [ ]:
from hanlp_restful import HanLPClient
from hanlp_common.document import Document
from time import sleep
import re

HanLP = HanLPClient('https://www.hanlp.com/api', auth="ODc1OEBiYnMuaGFubHAuY29tOlN5WVFNMFJrNnlqcm4yaGo=", language='zh')

def limit_sentense_length(text: str, max_length: int = 150) -> list[str]:
    if len(text) < 150:
        return [text]
    
    # print("Warning: 句子超出了长度上限: ", text)

    # 按照标点符号进行分割
    sentences = re.split(r'([。！？；，,.!?;])', text)
    chunks:list[str] = []
    temp = ''
    for i in range(0, len(sentences), 2):
        chunk = sentences[i] + (sentences[i+1] if i+1 < len(sentences) else '')
        if len(temp) + len(chunk) < max_length:
            temp += chunk
        else:
            chunks.append(temp)
            temp = chunk
    if temp:
        chunks.append(temp)
    return chunks

def split_texts(texts: list[str]) -> list[str]:
    """将文本列表中的每个文本分割成更小的句子"""
    all_sentences = []
    for text in texts:
        sentences = limit_sentense_length(text)
        all_sentences.extend(sentences)
    return all_sentences

def constituency_parsing_safe(texts: list[str], max_batch_num: int = 250, max_chars_per_batch: int = 15000, interval: int = 35) -> List[Document]:
    """对文本进行分词，同时限制每一批总字符数"""
    all_docs = []
    current_batch = []
    current_length = 0
    batch_id = 1

    for text in texts:
        text_len = len(text)

        # 如果加上这个句子会超出限制，则先处理已有批次
        if current_length + text_len > max_chars_per_batch or len(current_batch) + 1 > max_batch_num:
            print(f"Processing batch {batch_id} (Total chars: {current_length})...", end='')
            doc = HanLP.parse(current_batch, tasks=['pos', 'con'])
            all_docs.append(doc)
            print("done.")
            sleep(interval)

            batch_id += 1
            # 重置 batch
            current_batch = [text]
            current_length = text_len
        else:
            current_batch.append(text)
            current_length += text_len

    # 最后一批也别忘记
    if current_batch:
        print(f"Processing batch {batch_id} (Total chars: {current_length})...", end='')
        doc = HanLP.parse(current_batch, tasks=['pos', 'con'])
        all_docs.append(doc)
        print("done.")

    return all_docs

muice_responses_split = split_texts(muice_responses)
haruhi_responses_split = split_texts(haruhi_responses)
psydc_responses_split = split_texts(psydc_responses)

muice_cons = constituency_parsing_safe(muice_responses_split, 200, interval=17)
haruhi_cons = constituency_parsing_safe(haruhi_responses_split, 200, interval=18)
psydc_cons = constituency_parsing_safe(psydc_responses_split, 200, interval=18)

Processing batch 1 (Total chars: 11259)...done.
Processing batch 2 (Total chars: 11822)...done.
Processing batch 3 (Total chars: 10710)...done.
Processing batch 4 (Total chars: 12365)...done.
Processing batch 5 (Total chars: 12210)...done.
Processing batch 6 (Total chars: 11664)...done.
Processing batch 7 (Total chars: 12171)...done.
Processing batch 8 (Total chars: 11598)...done.
Processing batch 9 (Total chars: 11546)...done.
Processing batch 10 (Total chars: 11472)...done.
Processing batch 11 (Total chars: 11889)...done.
Processing batch 12 (Total chars: 12252)...done.
Processing batch 13 (Total chars: 12717)...done.
Processing batch 14 (Total chars: 11952)...done.
Processing batch 15 (Total chars: 12603)...done.
Processing batch 16 (Total chars: 11940)...done.
Processing batch 17 (Total chars: 11203)...done.
Processing batch 18 (Total chars: 11530)...done.
Processing batch 19 (Total chars: 11739)...done.
Processing batch 20 (Total chars: 12407)...done.
Processing batch 21 (Total ch

## 存档原始结果为 JSON 文件

In [ ]:
import json

muice_cons_json = [doc.to_dict() for doc in muice_cons]
with open('muice_cons.json', 'w', encoding='utf-8') as f:
    json.dump(muice_cons_json, f, ensure_ascii=False)

haruhi_cons_json = [doc.to_dict() for doc in haruhi_cons]
with open('haruhi_cons.json', 'w', encoding='utf-8') as f:
    json.dump(haruhi_cons_json, f, ensure_ascii=False)

psydc_cons_json = [doc.to_dict() for doc in psydc_cons]
with open('psydc_cons.json', 'w', encoding='utf-8') as f:
    json.dump(psydc_cons_json, f, ensure_ascii=False)


## 基于 Psydc 计算两种风格的对数似然比

In [ ]:
import json
import math
from typing import List, Dict, Tuple, Any, Optional, Literal
from collections import defaultdict, Counter


class PCFGExtractor:
    def __init__(self):
        self.rules_counter: Dict[str, Counter[Tuple[str, ...]]] = defaultdict(Counter)
        self.name: str = ""
        self.total_rules: int = 0

    def load_trees(self, file_path: str) -> List[Dict[str, Any]]:
        self.name = file_path
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)

    def extract_rules_from_tree(self, tree: Any):
        if not isinstance(tree, list) or len(tree) != 2:
            return
        lhs_symbol, rhs = tree
        if isinstance(rhs, list) and all(isinstance(child, list) and len(child) == 2 for child in rhs):
            rhs_symbols = tuple(child[0] for child in rhs)
            self.rules_counter[lhs_symbol][rhs_symbols] += 1
            self.total_rules += 1
            for child in rhs:
                self.extract_rules_from_tree(child)

    def extract_from_data(self, data: List[Dict[str, Any]]):
        for item in data:
            for tree in item.get("con", []):
                self.extract_rules_from_tree(tree)

    def build_pcfg(self) -> Dict[str, Dict[Tuple[str, ...], float]]:
        pcfg_distribution = {}
        for lhs_symbol, rhs_counter in self.rules_counter.items():
            total_count = sum(rhs_counter.values())
            pcfg_distribution[lhs_symbol] = {
                rhs: count / total_count for rhs, count in rhs_counter.items()
            }
        return pcfg_distribution

    def print_pcfg(
        self,
        pcfg: Dict[str, Dict[Tuple[str, ...], float]],
        sort_by: Literal["freq", "prob", "llr"] = 'freq',
        top_k: Optional[int] = None,
        baseline: Optional["PCFGExtractor"] = None,
        eps: float = 1e-5
    ):
        print(f"==={self.name} PCFG 产生式规则（按{'频率' if sort_by == 'freq' else ('对数似然比' if sort_by == 'llr' else '概率')}排序） ===")

        all_rules = []
        for lhs_symbol in self.rules_counter:
            for rhs_symbols in self.rules_counter[lhs_symbol]:
                freq = self.rules_counter[lhs_symbol][rhs_symbols]
                prob = pcfg[lhs_symbol][rhs_symbols]

                # PR / LLR
                pr = llr = None
                if baseline:
                    base_freq = baseline.rules_counter.get(lhs_symbol, {}).get(rhs_symbols, 0)
                    base_total = baseline.total_rules + eps
                    base_prob = base_freq / base_total

                    pr = (prob + eps) / (base_prob + eps)

                    k1, n1 = freq + eps, self.total_rules + eps
                    k2, n2 = base_freq + eps, base_total
                    mu = (k1 + k2) / (n1 + n2)
                    llr = 2 * (k1 * math.log(k1 / (n1 * mu)) + k2 * math.log(k2 / (n2 * mu)))

                all_rules.append((lhs_symbol, rhs_symbols, freq, prob, pr, llr))

        # 排序
        if sort_by == 'llr':
            all_rules.sort(key=lambda x: x[5] or 0, reverse=True)
        else:
            all_rules.sort(key=lambda x: x[2] if sort_by == 'freq' else x[3], reverse=True)

        # 打印
        for i, (lhs, rhs, freq, prob, pr, llr) in enumerate(all_rules):
            if top_k is not None and i >= top_k:
                break
            rhs_str = ' '.join(rhs)
            line = f"{lhs} → {rhs_str:<40} | freq={freq:<5} | P={prob:.4f}"
            if baseline:
                line += f" | PR={pr:.2f} | LLR={llr:.2f}"
            print(line)


def build_and_print_pcfg(file_path: str, baseline: Optional[PCFGExtractor] = None):
    extractor = PCFGExtractor()
    trees_data = extractor.load_trees(file_path)
    extractor.extract_from_data(trees_data)
    pcfg = extractor.build_pcfg()

    if baseline:
        extractor.print_pcfg(pcfg, sort_by='llr', top_k=15, baseline=baseline)
    else:
        extractor.print_pcfg(pcfg, sort_by='freq', top_k=15, baseline=baseline)
    print()

# 1. 先加载基准语料（psydc）
baseline_extractor = PCFGExtractor()
baseline_data = baseline_extractor.load_trees("psydc_cons.json")
baseline_extractor.extract_from_data(baseline_data)
baseline_extractor.build_pcfg()  # 可选，但为了接口统一性

# 2. 比较 muice / haruhi 与 psydc 的差异
build_and_print_pcfg("muice_cons.json", baseline=baseline_extractor)
build_and_print_pcfg("haruhi_cons.json", baseline=baseline_extractor)
build_and_print_pcfg("psydc_cons.json")  # 自身基准不做对比


===psydc_cons.json PCFG 产生式规则（按频率排序） ===
NP → PN                                       | freq=405362 | P=0.3313
NP → NN                                       | freq=384304 | P=0.3141
ADVP → AD                                       | freq=294804 | P=0.9250
IP → VP                                       | freq=272866 | P=0.3916
IP → NP VP                                    | freq=222805 | P=0.3197
VP → VV NP                                    | freq=180224 | P=0.1492
VP → ADVP VP                                  | freq=159355 | P=0.1319
VP → VV VP                                    | freq=127358 | P=0.1054
VP → VV                                       | freq=108984 | P=0.0902
NP → DNP NP                                   | freq=98342 | P=0.0804
VP → VV IP                                    | freq=93577 | P=0.0774
DNP → NP DEG                                   | freq=84483 | P=0.6032
VP → VA                                       | freq=80360 | P=0.0665
CP → IP DEC                          